# F1 score per field

Reads an `evaluation_results.jsonl` (one JSON object per evaluated image) and reports the
F1 score per extraction field, broken down by document type
(`BANK_STATEMENT`, `INVOICE`, `RECEIPT`).

There are two fieldwise tables, because averaging F1 has two defensible definitions and they
answer different questions:

| | macro table | micro table |
|---|---|---|
| unit of account | one document | one extracted item |
| computed as | mean of per-document F1 | F1 of the pooled tp/fp/fn |
| answers | "how good is a typical document?" | "what share of all transactions did we get?" |
| dominated by | nothing — equal weights | documents with long tables |

For single-valued fields (`DOCUMENT_TYPE`, `STATEMENT_DATE_RANGE`) the two coincide. For
list-valued fields they diverge, so always say which one you are quoting.

The macro table also carries `std_f1` (sample sd, ddof=1) and the `min_f1`/`max_f1` range.
Read the range first: these distributions tend to be a wall of 1.0s with a couple of total
failures, which is a shape the standard deviation describes badly.

In [ ]:
from pathlib import Path
import json

import pandas as pd

# Point this at the results file you want to analyse.
RESULTS_PATH = Path("/Users/tod/Desktop/evaluation_data/output/evaluation_results.jsonl")

In [ ]:
def load_field_scores(path: Path) -> pd.DataFrame:
    """Flatten an evaluation_results.jsonl into one row per (image, field).

    Args:
        path: Path to the JSONL results file.

    Returns:
        Long-format frame with image_name, document_type, field, f1_score, tp, fp, fn.
    """
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            for field, scores in record["field_scores"].items():
                rows.append(
                    {
                        "image_name": record["image_name"],
                        "document_type": record["document_type"],
                        "field": field,
                        "f1_score": scores["f1_score"],
                        "tp": scores["tp"],
                        "fp": scores["fp"],
                        "fn": scores["fn"],
                    }
                )
    if not rows:
        raise ValueError(f"No records with field_scores found in {path}")
    return pd.DataFrame(rows)


scores = load_field_scores(RESULTS_PATH)
print(f"{scores['image_name'].nunique()} documents, {scores['field'].nunique()} distinct fields")
print(scores.groupby("document_type")["image_name"].nunique().to_string())
scores.head()

In [ ]:
MACRO_COLS = ["n_docs", "mean_f1", "std_f1", "min_f1", "max_f1"]
MICRO_COLS = ["n_items", "tp", "fp", "fn", "micro_precision", "micro_recall", "micro_f1"]


def summarise(frame: pd.DataFrame, *, by: list[str]) -> pd.DataFrame:
    """Aggregate macro and micro F1 statistics over the given grouping columns.

    Args:
        frame: Long-format frame from load_field_scores.
        by: Columns to group on, e.g. ["document_type", "field"].

    Returns:
        One row per group holding both the MACRO_COLS and MICRO_COLS statistics.
        std_f1 is the sample standard deviation (ddof=1) of the per-document F1
        scores, so it is NaN for any group holding a single document.
    """
    summary = frame.groupby(by, dropna=False).agg(
        n_docs=("image_name", "nunique"),
        mean_f1=("f1_score", "mean"),
        std_f1=("f1_score", "std"),
        min_f1=("f1_score", "min"),
        max_f1=("f1_score", "max"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        fn=("fn", "sum"),
    )
    summary["n_items"] = summary["tp"] + summary["fn"]
    predicted = summary["tp"] + summary["fp"]
    actual = summary["tp"] + summary["fn"]
    summary["micro_precision"] = (summary["tp"] / predicted).where(predicted > 0, 0.0)
    summary["micro_recall"] = (summary["tp"] / actual).where(actual > 0, 0.0)
    denominator = 2 * summary["tp"] + summary["fp"] + summary["fn"]
    summary["micro_f1"] = (2 * summary["tp"] / denominator).where(denominator > 0, 0.0)
    return summary[MACRO_COLS + MICRO_COLS]


overall = summarise(scores, by=["field"]).sort_values("mean_f1")
per_type = summarise(scores, by=["document_type", "field"]).sort_values(
    ["document_type", "mean_f1"]
)
per_type.shape, overall.shape

## Macro table — every document counts once

`mean_f1` is the average of the per-document F1 scores. This is the number to quote when you
want "how well does the model do on a typical document". It is insensitive to table length,
so a one-line receipt and a 31-transaction bank statement carry equal weight.

In [ ]:
macro_table = overall[MACRO_COLS].round(4)
macro_table.insert(
    1,
    "mean_pm_std",
    overall.apply(lambda r: f"{r.mean_f1:.3f} ± {r.std_f1:.3f}", axis=1),
)

macro_by_type = per_type[MACRO_COLS].round(4)

display(macro_table)
display(macro_by_type)

## Micro table — every extracted item counts once

`micro_f1` pools the raw `tp`/`fp`/`fn` counts across documents before computing the score, so
a 31-transaction statement weighs 31x a single-transaction one. This is the number to quote for
"what fraction of all transactions in the corpus did we get right". `n_items` is the ground-truth
item count (`tp + fn`) behind each row — check it before trusting a field's micro score.

Precision and recall are broken out because they fail differently: low recall means truncated
extraction, low precision means hallucinated rows.

In [ ]:
micro_table = overall[MICRO_COLS].sort_values("micro_f1").round(4)
micro_by_type = per_type[MICRO_COLS].sort_values(["document_type", "micro_f1"]).round(4)

display(micro_table)
display(micro_by_type)

In [ ]:
# Mean F1 (and its spread) as a field x document-type matrix.
# NaN = field not evaluated for that document type.
matrix = per_type["mean_f1"].unstack("document_type")
errors = per_type["std_f1"].unstack("document_type")
matrix.round(4)

In [ ]:
# Error bars are +/- 1 standard deviation across documents, so they clip outside [0, 1].
ax = matrix.plot.barh(
    figsize=(9, 0.45 * len(matrix) + 2),
    xlim=(0, 1),
    xerr=errors.fillna(0.0),
    capsize=3,
    error_kw={"elinewidth": 1, "ecolor": "0.3"},
)
ax.set_xlabel("mean F1 (± 1 sd)")
ax.set_ylabel("")
ax.set_title(f"Mean F1 per field — {RESULTS_PATH.name}")
ax.legend(title="document type", loc="lower right")
ax.grid(axis="x", alpha=0.3)

In [ ]:
# Worst documents per field — the usual starting point for error analysis.
scores.sort_values("f1_score").head(15)[
    ["document_type", "field", "image_name", "f1_score", "tp", "fp", "fn"]
]